# HCDE 530 — Five questions: ChatGPT reviews (Kaggle)

**Dataset:** [ChatGPT reviews [DAILY UPDATED]](https://www.kaggle.com/datasets/ashishkumarak/chatgpt-reviews-daily-updated) by ashishkumarak.

## Getting the data (`kagglehub`)

Run the **next code cell**: `kagglehub.dataset_download(...)` downloads the dataset into Kaggle’s cache (for example under `~/.cache/kagglehub/datasets/.../versions/<number>/`) and stores that folder in the variable **`path`**.

The **loader cell after that** reads the CSV from `path` (searching subfolders with `rglob` if needed). You do **not** need to copy files into `Week5_project` unless you want a manual backup.

**Alternative:** unzip a CSV here as `chatgpt_reviews_kaggle.csv`, or run `kaggle datasets download ... --unzip` with API credentials—the loader falls back to this folder when `path` is not defined yet.

---

Below each section is a **plain-English comment** stating which checklist question it answers, then the pandas code.

In [3]:
import kagglehub

# Download latest version — saves cache dir string on `path` for the loader cell below.
path = kagglehub.dataset_download("ashishkumarak/chatgpt-reviews-daily-updated")

print("Path to dataset files:", path)

Path to dataset files: /Users/hza25/.cache/kagglehub/datasets/ashishkumarak/chatgpt-reviews-daily-updated/versions/634


In [4]:
from pathlib import Path

import pandas as pd

_EXCLUDE = {"app_info.csv", "app_reviews_demo.csv", "reviews_mini.csv"}


def load_chatgpt_reviews_csv(data_dir: Path) -> tuple[pd.DataFrame, Path]:
    """Load the main CSV under data_dir (kagglehub cache path or Week5_project)."""
    preferred = data_dir / "chatgpt_reviews_kaggle.csv"
    if preferred.exists():
        csv_path = preferred
    else:
        candidates = [
            p
            for p in data_dir.rglob("*.csv")
            if p.name not in _EXCLUDE and not p.name.startswith(".")
        ]
        if not candidates:
            raise FileNotFoundError(
                f"No CSV found under {data_dir}. Run the kagglehub download cell above first."
            )
        csv_path = max(candidates, key=lambda p: p.stat().st_size)
    return pd.read_csv(csv_path), csv_path


# Uses `path` from kagglehub.dataset_download(...) in the previous cell.
_root = globals().get("path")
dataset_root = Path(_root) if _root is not None else Path(".")

df, _csv_used = load_chatgpt_reviews_csv(dataset_root)
print("Loaded:", _csv_used.resolve())
print("Shape (rows, columns):", df.shape)
list(df.columns)

Loaded: /Users/hza25/.cache/kagglehub/datasets/ashishkumarak/chatgpt-reviews-daily-updated/versions/634/chatgpt_reviews.csv
Shape (rows, columns): (1019542, 8)


['reviewId',
 'userName',
 'content',
 'score',
 'thumbsUpCount',
 'reviewCreatedVersion',
 'at',
 'appVersion']

---
## Question 1 — What does the dataset look like?

Use **`head()`** and **`info()`** to see rows and column types.

In [5]:
# Question 1: What does this dataset look like? Show the first rows and column dtypes / non-null counts.
# Answer meaning: Confirms one row per review with text, star score, timestamps, and IDs — and shows typical content (short emoji posts vs longer complaints) before deeper analysis.
df.head()

,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion
0,2137dc90-83ab-4700-82ec-abed0f8b3e43,Laya Sri,Gpt 4 version was best. updated version is not...,1,0,1.2026.111,2026-04-28 09:30:53,1.2026.111
1,abfc784d-5427-4a4e-9789-04df20c7752f,Nasreen Bano,❤️❤️,5,0,1.2026.104,2026-04-28 09:30:43,1.2026.104
2,99597fa7-622f-40f6-ae90-2eee44935e66,S Elshamy,نور,5,0,NaN,2026-04-28 09:29:40,NaN
3,690e754b-8627-434b-8eca-0a29f33091cd,Manek Tahira,sooooooooo gooooooooooooooooooooood,5,0,1.2026.111,2026-04-28 09:29:16,1.2026.111
4,21448b44-ee5e-4950-b138-28b1c4c5b09d,Shopia Janet,I’ve been using this app consistently for the ...,5,0,1.2025.189,2026-04-28 09:25:29,1.2025.189


In [6]:
# Question 1 (continued): structure — dtypes, row count, missing counts per column at a glance.
# Answer meaning: Tells us scale (~1M rows), which fields are numeric vs text, and which columns already have gaps we must handle in MP1 (e.g. version fields).
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1019542 entries, 0 to 1019541
Data columns (total 8 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   reviewId              1019542 non-null  object
 1   userName              1019540 non-null  object
 2   content               1019526 non-null  object
 3   score                 1019542 non-null  int64 
 4   thumbsUpCount         1019542 non-null  int64 
 5   reviewCreatedVersion  945227 non-null   object
 6   at                    1019542 non-null  object
 7   appVersion            945227 non-null   object
dtypes: int64(2), object(6)
memory usage: 62.2+ MB


---
## Question 2 — Distribution of the most important column

For app reviews, **star rating** is usually the main outcome. We resolve the rating column whether Kaggle names it `rating`, `Rating`, `score`, etc.

In [7]:
# Question 2: What is the distribution of star ratings (the main numeric outcome for review sentiment)?
# Answer meaning: Shows whether opinions skew positive or negative — here most reviews are 5-star, so MP1 analyses should treat imbalance when comparing groups or sampling.
_cols_lower = {c.lower().strip(): c for c in df.columns}
_rating_candidates = ("rating", "score")
rating_col = next(
    (_cols_lower[k] for k in _rating_candidates if k in _cols_lower),
    None,
)
if rating_col is None:
    raise KeyError(f"Could not find a rating column among {list(df.columns)}")

df[rating_col].value_counts(dropna=False).sort_index()

score
1     67893
2     19004
3     42579
4    107082
5    782984
Name: count, dtype: int64

---
## Question 3 — Filter to a meaningful subset

Example: reviews with **low ratings** (1–2 stars), often used for qualitative inspection or complaint themes.

In [8]:
# Question 3: Focus on a meaningful subset — here, reviews with rating strictly below 3 (dissatisfied users).
# Answer meaning: Count + sample rows show how large the “unhappy” slice is and what kinds of text appear there — a pool for qualitative themes in MP1.
low_reviews = df[df[rating_col] < 3]
print(f"Subset size: {len(low_reviews)} rows")
low_reviews.head(10)

Subset size: 86897 rows


,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion
0,2137dc90-83ab-4700-82ec-abed0f8b3e43,Laya Sri,Gpt 4 version was best. updated version is not...,1,0,1.2026.111,2026-04-28 09:30:53,1.2026.111
5,45adcf99-617b-4d9d-95c4-02337e10c972,Nageswararao1,🙇‍♀️p😂😂qpa1qp p😂p pq😂q pq,1,0,1.2026.083,2026-04-28 09:24:30,1.2026.083
7,a9fe2b16-ed9d-4ea6-9155-ff7d84b9f9b8,S S,No to AI. BE ORIGINAL,1,0,NaN,2026-04-28 09:19:10,NaN
8,998cf33a-e2a6-4204-aae3-612c5c6ac50a,Abhishek Rai,very annoying,1,0,1.2026.104,2026-04-28 09:14:40,1.2026.104
11,359da645-3e39-48b3-8370-c950ae4a73e7,Manhattan cafe,Chat jewGPT,1,0,1.2026.111,2026-04-28 09:06:50,1.2026.111
15,1e96b42c-9092-429e-8cb7-f61386f8f284,Kdst Tadese,Qinbri,1,0,NaN,2026-04-28 09:02:01,NaN
17,cd8587e2-5c03-4c67-aa31-3b9fa8eb9786,Amir Hurani,save mother earth 🌎 save God's creation 🙏,1,0,1.2026.111,2026-04-28 09:00:41,1.2026.111
26,4c149e16-7a5d-4fcc-8f27-65b0d8fcdd9b,Corey,"its dumb as bricks now, gemeni is a far smarte...",1,0,1.2026.111,2026-04-28 08:48:16,1.2026.111
28,d4ce7f28-72d4-44d0-abc2-0f2c634897ea,Crypto King,killer,1,0,NaN,2026-04-28 08:47:45,NaN
31,66a48baf-7f13-466d-9175-51225841b1de,Tasnimul Hasan,খানকীর পোলা বাল বানাইছস,1,0,1.2026.111,2026-04-28 08:41:14,1.2026.111


---
## Question 4 — Group by a category; average of a numeric column

This CSV has no region field, so we **group by star rating** and summarize **average thumbs-up count** (one measure of how much other readers engaged with reviews at each rating level).

In [ ]:
# Question 4: Among star ratings, how does average “helpful” engagement (thumbs-up count) differ?
# Answer meaning: If lower-star reviews average many thumbs-ups, people may be amplifying complaints;
# if highs dominate, the metric may mostly reflect popularity of positive takes — worth comparing to text in MP1.
df.groupby(rating_col)["thumbsUpCount"].mean().sort_index().round(3)

---
## Question 5 — Where are missing values?

**`isnull().sum()`** counts missing cells per column.

In [10]:
# Question 5: Which columns are incomplete — how many missing values per column?
# Answer meaning: Points to fields we cannot trust for every row (e.g. app/version metadata) vs core fields that are almost complete — informs cleaning rules for Week 6 / MP1.
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

reviewCreatedVersion    74315
appVersion              74315
content                    16
userName                    2
dtype: int64

In [11]:
# Question 5 (supplement): total missing counts including columns that are fully complete — confirms zeros elsewhere.
df.isnull().sum()

reviewId                    0
userName                    2
content                    16
score                       0
thumbsUpCount               0
reviewCreatedVersion    74315
at                          0
appVersion              74315
dtype: int64